# Digitale Inline-Holografie mit OpenUC2 + ImSwitch

Lern-Notebook, das die Inline-Holografie erklaert und zeigt, wie der `InLineHoloController` von ImSwitch ueber das Python-Paket `imswitchclient` gesteuert wird.

**Lernziele**
1. Verbindung zu einem (per Docker bereitgestellten) ImSwitch-Backend    ueber `imswitchclient`.
2. Rohe Hologramm-Bilder von der Kamera holen.
3. Die Fresnel-Rekonstruktion lokal nachvollziehen.
4. Den Live-Prozessor im Backend ansteuern (Parameter, Start/Stop, ROI,    Binning).
5. Das rekonstruierte Hologramm als MJPEG-Livestream anzeigen.


## 1. Theorie in Kuerze

Ein *Inline*-Hologramm entsteht durch Interferenz zwischen der ungestreuten Beleuchtungswelle und der von der (meist transparenten) Probe gestreuten Welle. Die numerische Rekonstruktion propagiert das gemessene Intensitaetsbild ueber die Fresnel-Transferfunktion zurueck in die Probenebene:

$$U(x,y,z) = \mathcal{F}^{-1}\!\left\{\mathcal{F}\{U_0(x,y)\}\,e^{i k z}\,e^{i\pi\lambda z (f_x^2 + f_y^2)}\right\}$$

mit $k = 2\pi/\lambda$. Der Detektor misst nur die Intensitaet $I = |U|^2$. Mit dem richtigen Propagationsabstand `dz` wird das Objekt wieder scharf.


## 2. Abhaengigkeiten installieren und importieren


In [ ]:
# Install the ImSwitch client and plotting helpers (only needed once)
# Uncomment the next lines on a fresh environment:
# !pip install --quiet ipympl matplotlib numpy pillow requests
# !pip install --quiet https://github.com/openUC2/imswitchclient/archive/refs/heads/main.zip

import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image
import io
import time
import warnings
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Image as IPyImage

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)


## 3. Verbindung zu ImSwitch ueber den Client

`imswitchclient.ImSwitchClient` baut die Verbindung (HTTP oder HTTPS) auf und stellt Manager wie `recordingManager`, `positionersManager` usw. bereit. Zusaetzlich nutzen wir `client.get_json` / `client.post_json`, um die Endpunkte des `InLineHoloController` direkt aufzurufen.


In [ ]:
import imswitchclient.ImSwitchClient as imc

# --- Connection settings ---------------------------------------------------
# Change HOSTNAME / PORT to point at your ImSwitch backend.
# When ImSwitch runs in Docker on the same machine, "localhost" + port 80 works.
# For a remote machine, use its IP or DNS name (e.g. "imswitch.openuc2.com").
HOSTNAME = "localhost"
PORT     = 80
IS_HTTPS = False
ROUTE    = "/imswitch/api"

client = imc.ImSwitchClient(
    host=HOSTNAME,
    port=PORT,
    isHttps=IS_HTTPS,
    route=ROUTE,
)

# The base URI we will reuse for endpoints that the client does not yet wrap.
BASE_URI = client.base_uri
print("Connected to:", BASE_URI)


## 4. REST-Helfer fuer den InLineHoloController

Diese kleinen Wrapper bilden jeden Backend-Endpunkt auf eine Python-Funktion ab, damit der restliche Notebook-Code lesbar bleibt.


In [ ]:
# Thin wrappers around the InLineHoloController REST endpoints.
# The ImSwitch client exposes generic get_json / post_json helpers we can use
# to call any registered controller endpoint.

CTRL = "/InLineHoloController"

def holo_get_parameters():
    return client.get_json(f"{CTRL}/get_parameters_inlineholo")

def holo_set_parameters(**params):
    # set_parameters_inlineholo is a POST endpoint that expects {"params": {...}}
    return client.post_json(f"{CTRL}/set_parameters_inlineholo",
                            payload={"params": params})

def holo_set_dz(dz_m):
    return client.get_json(f"{CTRL}/set_dz_inlineholo", payload={"dz": dz_m})

def holo_set_wavelength(wavelength_m):
    return client.get_json(f"{CTRL}/set_wavelength_inlineholo",
                           payload={"wavelength": wavelength_m})

def holo_set_pixelsize(pixelsize_m):
    return client.get_json(f"{CTRL}/set_pixelsize_inlineholo",
                           payload={"pixelsize": pixelsize_m})

def holo_set_binning(binning):
    return client.get_json(f"{CTRL}/set_binning_inlineholo",
                           payload={"binning": int(binning)})

def holo_set_roi(center_x=None, center_y=None, size=256):
    payload = {"size": int(size)}
    if center_x is not None: payload["center_x"] = int(center_x)
    if center_y is not None: payload["center_y"] = int(center_y)
    return client.get_json(f"{CTRL}/set_roi_inlineholo", payload=payload)

def holo_set_update_freq(hz):
    return client.get_json(f"{CTRL}/set_update_freq_inlineholo",
                           payload={"update_freq": float(hz)})

def holo_start():
    return client.get_json(f"{CTRL}/start_processing_inlineholo")

def holo_stop():
    return client.get_json(f"{CTRL}/stop_processing_inlineholo")

def holo_pause():
    return client.get_json(f"{CTRL}/pause_processing_inlineholo")

def holo_resume():
    return client.get_json(f"{CTRL}/resume_processing_inlineholo")

def holo_state():
    return client.get_json(f"{CTRL}/get_state_inlineholo")

# URL of the processed-hologram MJPEG stream
HOLO_STREAM_URL = f"{BASE_URI}{CTRL}/mjpeg_stream_inlineholo?startStream=true&jpeg_quality=85"
print("Reconstruction stream URL:", HOLO_STREAM_URL)


## 5. Ein einzelnes Roh-Hologramm aufnehmen

Wir verwenden den Recording-Manager (bereits am Client verfuegbar), um ein einzelnes Bild als NumPy-Array von der Kamera zu holen.


In [ ]:
# Grab one raw hologram frame from the camera through the ImSwitch client.
# snapNumpyToFastAPI returns a 2D (mono) or 3D (RGB) numpy array.

raw_hologram = client.recordingManager.snapNumpyToFastAPI(resizeFactor=1)
print("Raw frame shape:", raw_hologram.shape, "dtype:", raw_hologram.dtype)

plt.figure()
plt.imshow(raw_hologram if raw_hologram.ndim == 2 else raw_hologram[..., 0],
           cmap="gray")
plt.title("Raw camera frame (hologram)")
plt.axis("off")
plt.show()


## 6. Lokale Fresnel-Rekonstruktion

Diese Python-Implementierung spiegelt den Algorithmus, der im Backend laeuft. Sie eignet sich fuer Offline-Experimente und zum Verstehen der Mathematik.


In [ ]:
class HologramProcessor:
    # Local Fresnel-propagation reconstruction, mirroring the backend math.

    def __init__(self, wavelength=488e-9, pixel_size=3.45e-6):
        self.wavelength = wavelength
        self.pixel_size = pixel_size
        self.k = 2 * np.pi / wavelength

    @staticmethod
    def abssqr(x):
        return np.real(x * np.conj(x))

    def _fft(self, x):  return np.fft.fftshift(np.fft.fft2(x))
    def _ifft(self, x): return np.fft.ifft2(np.fft.ifftshift(x))

    def fresnel_propagate(self, E0, dz):
        h, w = E0.shape
        Lx, Ly = self.pixel_size * w, self.pixel_size * h
        fx = np.linspace(-(w - 1) / 2 / Lx, (w - 1) / 2 / Lx, w)
        fy = np.linspace(-(h - 1) / 2 / Ly, (h - 1) / 2 / Ly, h)
        Fx, Fy = np.meshgrid(fx, fy)
        H = np.exp(1j * self.k * dz) * np.exp(
            1j * np.pi * self.wavelength * dz * (Fx ** 2 + Fy ** 2)
        )
        return self._ifft(H * self._fft(E0))

    def reconstruct(self, image, dz, crop=512):
        if image.ndim == 3:
            image = image.mean(axis=2)
        image = image.astype(np.float32)
        image /= max(image.max(), 1.0)
        h, w = image.shape
        c = min(crop, h, w)
        y0, x0 = (h - c) // 2, (w - c) // 2
        roi = image[y0:y0 + c, x0:x0 + c]
        field = self.fresnel_propagate(np.sqrt(roi), dz)
        return roi, self.abssqr(field), np.angle(field)


In [ ]:
# Pull current backend parameters and mirror them in a local processor.
params = holo_get_parameters()
print("Backend parameters:", params)

processor = HologramProcessor(
    wavelength=params.get("wavelength", 488e-9),
    pixel_size=params.get("pixelsize", 3.45e-6) * params.get("binning", 1),
)


In [ ]:
# Reconstruct the snapped frame locally for different propagation distances.
dz_values_mm = [1.0, 3.0, 5.0, 8.0]

fig, axes = plt.subplots(1, len(dz_values_mm), figsize=(4 * len(dz_values_mm), 4))
for ax, dz_mm in zip(axes, dz_values_mm):
    _, intensity, _ = processor.reconstruct(raw_hologram, dz_mm * 1e-3, crop=512)
    ax.imshow(intensity, cmap="gray")
    ax.set_title(f"dz = {dz_mm:.1f} mm")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 7. Interaktive Slider (lokal + remote)

Jede Aenderung wird per `set_parameters_inlineholo` ans Backend geschickt *und* aktualisiert die lokale Vorschau. Der MJPEG-Stream weiter unten reagiert dadurch in Echtzeit.


In [ ]:
# Interactive sliders that update BOTH the local processor and the
# remote backend (so the live MJPEG stream below reacts in real time).

dz_slider = widgets.FloatSlider(value=params.get("dz", 0.0) * 1e3,
                                min=-10, max=20, step=0.1,
                                description="dz [mm]")
wl_slider = widgets.FloatSlider(value=params.get("wavelength", 488e-9) * 1e9,
                                min=380, max=750, step=5,
                                description="lambda [nm]")
px_slider = widgets.FloatSlider(value=params.get("pixelsize", 3.45e-6) * 1e6,
                                min=0.5, max=10.0, step=0.05,
                                description="px [um]")
bin_dd   = widgets.Dropdown(options=[1, 2, 4, 8],
                            value=params.get("binning", 1),
                            description="binning")
roi_size = widgets.IntSlider(value=params.get("roi_size", 256),
                             min=64, max=1024, step=32,
                             description="ROI size")
out      = widgets.Output()

def on_change(_):
    with out:
        clear_output(wait=True)
        # 1) push to the backend so the live stream reflects the changes
        holo_set_parameters(
            dz=dz_slider.value * 1e-3,
            wavelength=wl_slider.value * 1e-9,
            pixelsize=px_slider.value * 1e-6,
            binning=int(bin_dd.value),
            roi_size=int(roi_size.value),
        )
        # 2) recompute locally for the static preview
        processor.wavelength = wl_slider.value * 1e-9
        processor.pixel_size = px_slider.value * 1e-6 * int(bin_dd.value)
        processor.k = 2 * np.pi / processor.wavelength
        _, intensity, phase = processor.reconstruct(
            raw_hologram, dz_slider.value * 1e-3, crop=int(roi_size.value)
        )
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].imshow(intensity, cmap="gray"); ax[0].set_title("|E|^2"); ax[0].axis("off")
        ax[1].imshow(phase, cmap="twilight");  ax[1].set_title("phase"); ax[1].axis("off")
        plt.tight_layout(); plt.show()

for w in (dz_slider, wl_slider, px_slider, bin_dd, roi_size):
    w.observe(on_change, names="value")

display(widgets.VBox([dz_slider, wl_slider, px_slider, bin_dd, roi_size, out]))
on_change(None)


## 8. Backend-Verarbeitung starten


In [ ]:
# Start the backend processing pipeline.
print(holo_start())
print(holo_set_update_freq(10.0))
print("State:", holo_state())


## 9. Live-Rekonstruktions-Stream

Der Endpunkt `mjpeg_stream_inlineholo` liefert einen kontinuierlichen Strom rekonstruierter Bilder. Aenderungen an den Slidern oben werden sofort sichtbar.


In [ ]:
# Embed the processed-hologram MJPEG stream directly in the notebook.
# (Most browsers will keep refreshing the <img> tag automatically.)
display(HTML(f'<img src="{HOLO_STREAM_URL}" style="max-width:640px;border:1px solid #888"/>'))


## 10. Backend am Ende stoppen


In [ ]:
# Pause / resume / stop are mapped to the corresponding REST calls.
# print(holo_pause())
# print(holo_resume())
print(holo_stop())


## 11. Uebungen

1. Variiere `dz` zwischen -5 mm und +15 mm und finde visuell den Fokus.
2. Reduziere `binning` auf 1 und erhoehe `roi_size` - wie veraendert
   sich die Verarbeitungsrate?
3. Spiegle das Bild (`flip_x`/`flip_y`) und rotiere es ueber die
   entsprechenden Schluessel in `holo_set_parameters`.
4. Nimm zwei Schnappschuesse vor und nach einer Fokusaenderung auf
   und vergleiche sie mit der lokalen Rekonstruktion.


---
*Erstellt fuer die OpenUC2-Community. Viel Spass beim Holografieren!*
